# Silver — `contas`

## Bibliotecas

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

## Variáveis

In [0]:
# origem
bronze_contas   = "novarota.bronze.contas"
silver_clientes = "novarota.silver.clientes"   # pra conferir se o cliente existe

# destino
tabela_silver     = "novarota.silver.contas"
tabela_quarentena = "novarota.silver.quarentena_contas"
tabela_controle   = "novarota.silver.controle_cargas"

# chave negocio
chaves_negocio = ["id_conta"]

print(f"Tabela alvo: {tabela_silver}")

## Ultima carga

In [0]:
# cargas que existem na bronze
cargas_bronze = [l.data_referencia for l in spark.table(bronze_contas).select("data_referencia").distinct().collect()]

# cargas que a silver de contas já processou
if spark.catalog.tableExists(tabela_controle):
    cargas_processadas = [
        l.data_referencia
        for l in spark.table(tabela_controle).filter("tabela = 'contas'").select("data_referencia").distinct().collect()
    ]
else:
    cargas_processadas = []

cargas_pendentes = sorted(c for c in cargas_bronze if c not in cargas_processadas)

print(f"Cargas já processadas: {sorted(cargas_processadas)}")
print(f"Cargas pendentes: {cargas_pendentes}")

if not cargas_pendentes:
    dbutils.notebook.exit("Nada pendente na bronze de contas")

## Transformações

In [0]:
filtro_incremental = "where data_referencia in (" + ", ".join(f"'{c}'" for c in cargas_pendentes) + ")"

query = f"""
with contas as (select *, to_json(struct(*)) as registro_original
from {bronze_contas} {filtro_incremental})
,
validado as (
select
    a.id_conta,
    a.id_cliente,
    a.tipo_conta,
    upper(trim(a.status_conta))       as status_conta,
    try_cast(a.data_abertura as date) as data_abertura,
    a.data_referencia,
    a.timestamp_ingestao,
    case
        when a.id_conta is null then 'CHAVE_NULA'
        when coalesce(upper(trim(a.status_conta)), '') not in ('ATIVA', 'BLOQUEADA', 'ENCERRADA') then 'STATUS_INVALIDO'
        when b.id_cliente is null then 'CLIENTE_INEXISTENTE'
    end as motivo_erro,
    a.registro_original
from contas a
left join {silver_clientes} b on a.id_cliente = b.id_cliente)


select
    *,
    row_number() over (
        partition by id_conta
        order by case when motivo_erro is null then 0 else 1 end,
                 data_referencia desc,
                 timestamp_ingestao desc
    ) as rn
from validado
"""

df = spark.sql(query)

print(f"Linhas lidas da origem (cargas pendentes): {df.count()}")

## Válidos x quarentena

In [0]:
# válidos: sem motivo de erro e só a versão mais recente de cada conta
df_validos = (
    df.filter("motivo_erro is null and rn = 1")
      .drop("motivo_erro", "registro_original", "rn")
)

# quarentena: tudo que tem motivo de erro, com a linha como chegou da bronze
df_quarentena = (
    df.filter("motivo_erro is not null")
      .select(
          F.col("id_conta").alias("chave"),
          "motivo_erro",
          "registro_original",
          "data_referencia",
          F.current_timestamp().alias("timestamp_quarentena"),
      )
)

print(f"Válidos (após dedup): {df_validos.count()}")
print(f"Quarentena: {df_quarentena.count()}")

## Conferência

In [0]:
dups_no_lote = (
    df_validos.groupBy(*chaves_negocio).count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Chaves duplicadas no lote atual (esperado 0): {dups_no_lote}")

df_quarentena.display()

## Carga incremental — MERGE

In [0]:
if not spark.catalog.tableExists(tabela_silver):
    (
        df_validos.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela_silver)
    )
    print(f"Tabela criada (carga inicial): {tabela_silver} | {df_validos.count()} linhas")
else:
    tabela = DeltaTable.forName(spark, tabela_silver)
    condicao = " AND ".join([f"t.{k} = s.{k}" for k in chaves_negocio])

    (
        tabela.alias("t")
        .merge(df_validos.alias("s"), condicao)
        # contas não tem data_atualizacao: a versão mais nova é a da carga mais recente
        .whenMatchedUpdateAll(condition="s.data_referencia > t.data_referencia")
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE concluído em {tabela_silver}")

## Quarentena e controle

In [0]:
# quarentena: só acumula, cada carga é processada uma vez
df_quarentena.write.format("delta").mode("append").saveAsTable(tabela_quarentena)

# controle: uma linha por carga processada. Fica por último, assim a carga só conta como processada se tudo antes deu certo
controle = (
    df.groupBy("data_referencia")
      .agg(
          F.sum(F.when(F.col("motivo_erro").isNull(), 1).otherwise(0)).alias("linhas_validas"),
          F.sum(F.when(F.col("motivo_erro").isNotNull(), 1).otherwise(0)).alias("linhas_quarentena"),
      )
      .select(
          F.lit("contas").alias("tabela"),
          "data_referencia",
          F.col("linhas_validas").cast("bigint"),
          F.col("linhas_quarentena").cast("bigint"),
          F.current_timestamp().alias("processado_em"),
      )
)
controle.write.format("delta").mode("append").saveAsTable(tabela_controle)

controle.display()

## Conferência pós-escrita

In [0]:
print(f"Conferência da tabela: {tabela_silver}")

df_silver = spark.table(tabela_silver)
print(f"Total de linhas: {df_silver.count()}")

dups = (
    df_silver.groupBy(*chaves_negocio).count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Chaves duplicadas (esperado 0): {dups}")

df_silver.limit(50).display()